# Run 3 (2024) data / MC comparison

Stacked MC vs data for the control variables of the analysis selection (`btag_mask` category),
**before any Xbb cut** by default. Reads the merged inference outputs
(`outputs/nominal/<group>/output_all.coffea`, from `run_pipeline.py inference`), one file at a
time, and fills the histograms directly (no pickles needed).

MC weight = `norm_weight` (xsec × lumi) × sign(genWeight) / Σ sign(genWeight) × the SFs of
`outvars.nominal_weight_factors`. Until they are derived, the electron trigger SF and the b-tag
SF are 1 and the muon trigger SF is the MUO reference (see README), so the agreement is not final.

Uncertainty band: MC statistics ⊕ the weight variations stored in the columns (per source, the
larger of |up − nom|, |down − nom| summed over processes, added in quadrature).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplhep as hep
from matplotlib.backends.backend_pdf import PdfPages
from coffea.util import load

import outvars
from analysis_years import LUMI_FB, xbb_wp_cuts

hep.style.use(hep.style.CMS)

# ---------------- USER SETTINGS ----------------
ERA = "2024"
OUTPUT_DIR = "outputs/nominal"                       # run_pipeline.py layout
GROUPS = ["signal", "bkg", "minor", "data"]         # add "qcd" together with INCLUDE_QCD
CATEGORY = "btag_mask"
INCLUDE_QCD = False                                  # negligible; not part of the analysis

# Event selection on top of the btag_mask category (None disables a requirement)
SELECTION = {
    "min_ak4_jets": 5,          # n_ak4jets >=
    "n_b_outZH": 2,             # b jets outside the candidate ==
    "zh_mass": (50, 200),       # GeV
    "min_zh_pt": 200,           # GeV
}
XBB_WP = None                   # None (no Xbb cut) or "LP"/"MP"/"HP" once analysis_years.XBB_WP_DICT is filled
CHANNEL = "all"                 # "all", "ele" or "mu" (tight-lepton flavour)

# Data hidden above these values (signal-like region); {} to unblind
BLIND = {"ZH_bbvLscore": 0.8}

TTB_SCALE = 1.0                 # extra scale of the 4FS tt+B yield (1 = cross sections as in the datasets)
PLOT_DIR = "plots/datamc"
# -----------------------------------------------

LUMI = LUMI_FB[ERA]
TAG = f"{ERA}_{CHANNEL}_" + (f"Xbb{XBB_WP}" if XBB_WP else "NoXbb")
os.makedirs(PLOT_DIR, exist_ok=True)
print(f"Era {ERA}, {LUMI} fb^-1, channel {CHANNEL}, Xbb cut {XBB_WP}, output tag {TAG}")

In [ ]:
# ---------------- variables and binning: name -> (bins, xlabel, log y)
VARIABLES = {
    "nPV":          (np.linspace(0, 80, 41),  "Number of PVs", False),
    "MET_pt":       (np.linspace(0, 500, 41), r"PUPPI $p_{T}^{miss}$ [GeV]", True),
    "MET_phi":      (np.linspace(-np.pi, np.pi, 31), r"PUPPI $\phi^{miss}$", False),
    "lep_pt":       (np.linspace(0, 500, 41), r"Lepton $p_{T}$ [GeV]", True),
    "lep_eta":      (np.linspace(-2.5, 2.5, 31), r"Lepton $\eta$", False),
    "ele_pt":       (np.linspace(0, 500, 41), r"Electron $p_{T}$ [GeV]", True),
    "ele_eta":      (np.linspace(-2.5, 2.5, 31), r"Electron $\eta$", False),
    "muon_pt":      (np.linspace(0, 500, 41), r"Muon $p_{T}$ [GeV]", True),
    "muon_eta":     (np.linspace(-2.4, 2.4, 31), r"Muon $\eta$", False),
    "n_ak4":        (np.arange(4.5, 15.5, 1), "Number of AK4 jets", True),
    "n_bjet":       (np.arange(-0.5, 8.5, 1), "Number of b jets (UParT M)", True),
    "n_ak8":        (np.arange(0.5, 5.5, 1), "Number of AK8 jets", True),
    "jet1_pt":      (np.linspace(0, 600, 41), r"Leading AK4 jet $p_{T}$ [GeV]", True),
    "jet2_pt":      (np.linspace(0, 500, 41), r"Subleading AK4 jet $p_{T}$ [GeV]", True),
    "jet1_eta":     (np.linspace(-2.5, 2.5, 31), r"Leading AK4 jet $\eta$", False),
    "jet2_eta":     (np.linspace(-2.5, 2.5, 31), r"Subleading AK4 jet $\eta$", False),
    "jet1_btag":    (np.linspace(0, 1, 21), "Leading AK4 jet UParT B", True),
    "jet2_btag":    (np.linspace(0, 1, 21), "Subleading AK4 jet UParT B", True),
    "bjet1_pt":     (np.linspace(0, 500, 41), r"Leading b jet $p_{T}$ [GeV]", True),
    "bjet1_eta":    (np.linspace(-2.5, 2.5, 31), r"Leading b jet $\eta$", False),
    "bjet1_btag":   (np.linspace(0, 1, 21), "Leading b jet UParT B", True),
    "fatjet1_pt":   (np.linspace(200, 800, 31), r"Leading AK8 jet $p_{T}$ [GeV]", True),
    "fatjet1_eta":  (np.linspace(-2.5, 2.5, 31), r"Leading AK8 jet $\eta$", False),
    "fatjet1_mass": (np.linspace(50, 200, 31), "Leading AK8 jet GloParT mass [GeV]", False),
    "ZH_pt":        (np.linspace(200, 800, 31), r"Z/H candidate $p_{T}$ [GeV]", True),
    "ZH_M":         (np.linspace(50, 200, 31), "Z/H candidate GloParT mass [GeV]", False),
    "ZH_bbvLscore": (np.linspace(0, 1, 21), "Z/H candidate GloParT XbbVsQCD", True),
    "n_b_inZH":     (np.arange(-0.5, 4.5, 1), "b jets inside the candidate", True),
    "n_ak4jets":    (np.arange(4.5, 15.5, 1), "AK4 jets", True),
}

# ---------------- processes: stack order (bottom -> top), labels, colors
# Colors: fixed categorical order (colorblind-validated palette); signal on top of the stack
PROCESSES = ["TTBar", "tt_B", "SingleTop", "ttX", "VJets", "Other", "ttZ", "ttH"]
LABELS = {"TTBar": r"$t\bar{t}$+LF/C", "tt_B": r"$t\bar{t}$+B", "SingleTop": "Single top",
          "ttX": r"$t\bar{t}$+X", "VJets": "V+jets", "Other": "VV" + (", QCD" if INCLUDE_QCD else ""),
          "ttZ": r"$t\bar{t}Z$", "ttH": r"$t\bar{t}H$"}
COLORS = dict(zip(PROCESSES, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4",
                              "#008300", "#4a3aa7", "#e34948"]))


def process_of(key):
    """Output key (sample or sample__subsample) -> process; None = not used.
    tt+B comes from the 4FS TTbb samples, tt+LF/C from the 5FS TTTo samples."""
    sample, _, sub = key.partition("__")
    if sample.startswith("DATA"):
        return "data"
    if sample.startswith("TTbb"):
        return "tt_B" if sub == "tt+B" else None
    if sample.startswith("TTTo"):
        return "TTBar" if sub in ("tt+LF", "tt+C") else None
    if sample in ("ttHTobb", "ttHToNonbb"):
        return "ttH"
    if sample in ("TTZToQQ", "TTLL", "TTNuNu"):
        return "ttZ"
    if sample in ("WJets", "DYJets"):
        return "VJets"
    if sample == "SingleTop":
        return "SingleTop"
    if sample == "TTX":
        return "ttX"
    if sample == "VV" or (sample == "QCD" and INCLUDE_QCD):
        return "Other"
    return None


# ---------------- weights
NOMINAL_FACTORS = list(outvars.nominal_weight_factors)
# source -> (nominal column or None, up, down); trigger and b-tag SFs are 1 until derived
WEIGHT_SYSTS = {
    "topptWeight": ("topptWeight", "topptWeight_Up", "topptWeight_Down"),
    "ele_reco":    ("ele_reco_sf", "ele_reco_sfup", "ele_reco_sfdown"),
    "ele_id":      ("ele_id_sf", "ele_id_sfup", "ele_id_sfdown"),
    "ele_trig":    ("ele_trig_sf", "ele_trig_sfup", "ele_trig_sfdown"),
    "mu_id":       ("mu_id_sf", "mu_id_sfup", "mu_id_sfdown"),
    "mu_iso":      ("mu_iso_sf", "mu_iso_sfup", "mu_iso_sfdown"),
    "mu_trig":     ("mu_trig_sf", "mu_trig_sfup", "mu_trig_sfdown"),
    "btag":        ("btag_sf", "btag_sfup", "btag_sfdown"),
    "pileup":      ("puWeight", "puWeight_up", "puWeight_down"),
    "isr":         (None, "isr_up", "isr_down"),
    "fsr":         (None, "fsr_up", "fsr_down"),
    "mu_r":        (None, "mu_r_up", "mu_r_down"),
    "mu_f":        (None, "mu_f_up", "mu_f_down"),
    "mu_rf":       (None, "mu_rf_up", "mu_rf_down"),
}

CUT_VARS = ["n_ak4jets", "n_b_outZH", "ZH_M", "ZH_pt", "ZH_bbvLscore", "ele_pt", "muon_pt"]
MC_COLUMNS = (["genWeight", "norm_weight"] + NOMINAL_FACTORS
              + [c for t in WEIGHT_SYSTS.values() for c in t if c])


def selection_mask(df):
    m = np.ones(len(df), dtype=bool)
    s = SELECTION
    if s.get("min_ak4_jets") is not None:
        m &= df["n_ak4jets"].to_numpy() >= s["min_ak4_jets"]
    if s.get("n_b_outZH") is not None:
        m &= df["n_b_outZH"].to_numpy() == s["n_b_outZH"]
    if s.get("zh_mass") is not None:
        m &= (df["ZH_M"].to_numpy() >= s["zh_mass"][0]) & (df["ZH_M"].to_numpy() <= s["zh_mass"][1])
    if s.get("min_zh_pt") is not None:
        m &= df["ZH_pt"].to_numpy() >= s["min_zh_pt"]
    if XBB_WP is not None:
        m &= df["ZH_bbvLscore"].to_numpy() >= xbb_wp_cuts(ERA)[XBB_WP[0]]
    if CHANNEL == "ele":
        m &= df["ele_pt"].to_numpy() > 0
    elif CHANNEL == "mu":
        m &= df["muon_pt"].to_numpy() > 0
    return m


def mc_weight(df, sum_sign_genw):
    w = df["norm_weight"].to_numpy() * np.sign(df["genWeight"].to_numpy()) / sum_sign_genw
    for f in NOMINAL_FACTORS:
        if f in df:
            w = w * df[f].to_numpy()
    return w

In [ ]:
# ---------------- read the outputs one file at a time and fill the histograms
# hists[var][process][variation] = (sumw, sumw2); variation = "nominal" or "<source>Up/Down"
hists = {v: {} for v in VARIABLES}
yields = {}
missing_vars = set()


def fill(proc, df, w, variation):
    for var, (bins, _, _) in VARIABLES.items():
        if var not in df:
            missing_vars.add(var)
            continue
        x = df[var].to_numpy()
        ok = x > -998                                   # -999 = not defined (e.g. no electron)
        xv = np.clip(x[ok], bins[0], bins[-1] - 1e-9)  # under/overflow into the edge bins
        sumw, _ = np.histogram(xv, bins, weights=w[ok])
        sumw2, _ = np.histogram(xv, bins, weights=w[ok] ** 2)
        h = hists[var].setdefault(proc, {})
        if variation in h:
            h[variation] = (h[variation][0] + sumw, h[variation][1] + sumw2)
        else:
            h[variation] = (sumw, sumw2)


for group in GROUPS:
    path = os.path.join(OUTPUT_DIR, group, "output_all.coffea")
    if not os.path.exists(path):
        print(f"[skip] {path} not found")
        continue
    print(f"Reading {path}")
    out = load(path)
    for key, datasets in out["columns"].items():
        proc = process_of(key)
        if proc is None:
            continue
        for dataset, cats in datasets.items():
            if ERA not in dataset:
                continue
            cols = cats.get(CATEGORY, {}).get("nominal")
            if not cols:
                continue
            names = set(VARIABLES) | set(CUT_VARS) | (set() if proc == "data" else set(MC_COLUMNS))
            df = pd.DataFrame({n: cols[f"events_{n}"].value for n in names if f"events_{n}" in cols})
            df = df[selection_mask(df)]
            if proc == "data":
                w = np.ones(len(df))
                fill(proc, df, w, "nominal")
            else:
                w = mc_weight(df, out["sum_signOf_genweights"][dataset])
                if proc == "tt_B":
                    w = w * TTB_SCALE
                fill(proc, df, w, "nominal")
                for src, (nom, up, dn) in WEIGHT_SYSTS.items():
                    if up not in df or dn not in df:
                        continue
                    base = df[nom].to_numpy() if nom else np.ones(len(df))
                    base = np.where(base == 0, 1.0, base)
                    fill(proc, df, w * df[up].to_numpy() / base, f"{src}Up")
                    fill(proc, df, w * df[dn].to_numpy() / base, f"{src}Down")
            yields.setdefault(proc, [0.0, 0.0, 0])
            yields[proc][0] += w.sum(); yields[proc][1] += (w ** 2).sum(); yields[proc][2] += len(df)
    del out

if missing_vars:
    print(f"Variables not in the outputs (skipped): {sorted(missing_vars)}")

In [ ]:
# ---------------- yield table
rows = []
for proc in PROCESSES + ["data"]:
    if proc in yields:
        sw, sw2, n = yields[proc]
        rows.append({"process": LABELS.get(proc, "Data"), "yield": sw, "stat. unc.": np.sqrt(sw2), "raw events": n})
table = pd.DataFrame(rows).set_index("process")
mc_total = sum(yields[p][0] for p in PROCESSES if p in yields)
data_total = yields.get("data", [0, 0, 0])[0]
print(table.round(2).to_string())
print(f"\nTotal MC: {mc_total:.1f}   Data: {data_total:.0f}   Data/MC: "
      f"{data_total / mc_total if mc_total else float('nan'):.3f}")
table.to_csv(os.path.join(PLOT_DIR, f"yields_{TAG}.csv"))

In [ ]:
# ---------------- plotting
SOURCES = list(WEIGHT_SYSTS)


def mc_band(var, total):
    """Stat ⊕ weight systematics of the MC total."""
    stat2 = sum(hists[var][p]["nominal"][1] for p in PROCESSES if p in hists[var])
    syst2 = np.zeros_like(total)
    for src in SOURCES:
        up = np.zeros_like(total); dn = np.zeros_like(total)
        for proc in PROCESSES:
            h = hists[var].get(proc)
            if not h:
                continue
            nom = h["nominal"][0]
            up += h.get(f"{src}Up", h["nominal"])[0] - nom
            dn += h.get(f"{src}Down", h["nominal"])[0] - nom
        syst2 += np.maximum(np.abs(up), np.abs(dn)) ** 2
    return np.sqrt(stat2 + syst2)


def plot_var(var):
    bins, xlabel, logy = VARIABLES[var]
    procs = [p for p in PROCESSES if p in hists[var]]
    if not procs:
        return None
    fig, (ax, rax) = plt.subplots(2, 1, figsize=(10, 10), gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05},
                                  sharex=True)
    stack = [hists[var][p]["nominal"][0] for p in procs]
    total = np.sum(stack, axis=0)
    err = mc_band(var, total)
    hep.histplot(stack, bins, ax=ax, stack=True, histtype="fill", color=[COLORS[p] for p in procs],
                 label=[f"{LABELS[p]} ({s.sum():.4g})" for p, s in zip(procs, stack)],
                 edgecolor="white", linewidth=0.5)
    ax.stairs(total + err, bins, baseline=np.clip(total - err, 0, None), fill=True, hatch="////",
              facecolor="none", edgecolor="#555555", linewidth=0, label=r"Stat $\oplus$ syst")

    data = hists[var].get("data", {}).get("nominal")
    centers = 0.5 * (bins[1:] + bins[:-1])
    if data is not None:
        d = data[0].astype(float)
        if var in BLIND:
            d[bins[1:] > BLIND[var]] = np.nan
        hep.histplot(d, bins, ax=ax, histtype="errorbar", yerr=np.sqrt(d), color="black",
                     label=f"Data ({np.nansum(d):.0f})")
        pos = total > 0                                 # bins without positive MC: no ratio
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = np.where(pos, d / total, np.nan)
            rerr = np.where(pos, np.sqrt(d) / total, np.nan)
        rax.errorbar(centers, ratio, yerr=rerr, fmt="o", color="black", markersize=4)
    with np.errstate(divide="ignore", invalid="ignore"):
        rel = np.where(total > 0, err / total, 0)
    rax.stairs(1 + rel, bins, baseline=1 - rel, fill=True, hatch="////", facecolor="none",
               edgecolor="#555555", linewidth=0)
    rax.axhline(1, color="black", lw=1, ls="--")
    rax.set_ylim(0.5, 1.5)
    rax.set_ylabel("Data / MC")
    rax.set_xlabel(xlabel)
    ax.set_ylabel("Events")
    top = max(total.max(), np.nanmax(d) if data is not None and np.isfinite(d).any() else 0)
    if logy:
        ax.set_yscale("log")
        pos = np.concatenate([total[total > 0], d[np.isfinite(d) & (d > 0)] if data is not None else []])
        ax.set_ylim(pos.min() * 0.5 if len(pos) else 0.1, top * 1e3)
    else:
        ax.set_ylim(0, top * 1.6)
    ax.legend(ncol=2, fontsize="x-small", loc="upper right")
    hep.cms.label("Preliminary", data=data is not None, lumi=LUMI, com=13.6, ax=ax)
    return fig


with PdfPages(os.path.join(PLOT_DIR, f"datamc_{TAG}.pdf")) as pdf:
    for var in VARIABLES:
        fig = plot_var(var)
        if fig is None:
            continue
        pdf.savefig(fig, bbox_inches="tight")
        fig.savefig(os.path.join(PLOT_DIR, f"{var}_{TAG}.png"), bbox_inches="tight", dpi=120)
        plt.close(fig)
print(f"Plots in {PLOT_DIR}: datamc_{TAG}.pdf and one PNG per variable")

In [ ]:
# Show a few plots inline
for var in ["MET_pt", "lep_pt", "ZH_M", "ZH_bbvLscore"]:
    fig = plot_var(var)
    if fig is not None:
        plt.show()